# Merge 2018-2019 Floors 3, 4, 5, and 7

This notebook is a focused version of the merge workflow. It only merges the raw CSV files for floors 3, 4, 5, and 7 across 2018 and 2019.

Scope:

- Years: `2018`, `2019`
- Floors: `3`, `4`, `5`, `7`
- Input folder: `Raw Data/`
- Output folder: `outputs/`
- Main output: `combined_wide_2018_2019_floors_3_4_5_7.csv.gz`

The merge is vertical concatenation. It keeps `year`, `floor`, and `source_file` columns so every row can be traced back to its original CSV file.


## 1. Import libraries and locate project folders


In [ ]:
from pathlib import Path
from typing import Optional
import re

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_NAME = "Artificial Intelligence for Smart-Building Log Analysis"
YEARS = [2018, 2019]
FLOORS = [3, 4, 5, 7]


def find_project_dir(start: Optional[Path] = None) -> Path:
    """Find the project folder from the current notebook location or parent folders."""
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if path.name == PROJECT_NAME and (path / "Raw Data").exists():
            return path
        candidate = path / PROJECT_NAME
        if (candidate / "Raw Data").exists():
            return candidate
    raise FileNotFoundError(f"Could not find '{PROJECT_NAME}/Raw Data'.")


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "Raw Data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_DIR, DATA_DIR, OUTPUT_DIR


## 2. Select the required 8 source CSV files


In [ ]:
source_files = [DATA_DIR / f"{year}Floor{floor}.csv" for year in YEARS for floor in FLOORS]
missing_files = [path for path in source_files if not path.exists()]

if missing_files:
    raise FileNotFoundError("Missing required CSV files:\n" + "\n".join(str(path) for path in missing_files))

source_file_table = pd.DataFrame({
    "year": [int(re.search(r"(20\d{2})", path.name).group(1)) for path in source_files],
    "floor": [int(re.search(r"Floor(\d+)", path.name).group(1)) for path in source_files],
    "file_name": [path.name for path in source_files],
    "file_path": [str(path) for path in source_files],
})

source_file_table


## 3. Check column counts and field consistency

Floors 3, 4, 5, and 7 should use the same 30 fields. This check prevents accidental merging by position when field names do not match.


In [ ]:
headers = {}
for path in source_files:
    headers[path.name] = list(pd.read_csv(path, nrows=0).columns)

reference_file = source_files[0].name
reference_columns = headers[reference_file]

column_check = pd.DataFrame({
    "file_name": list(headers.keys()),
    "column_count": [len(cols) for cols in headers.values()],
    "same_columns_and_order_as_reference": [cols == reference_columns for cols in headers.values()],
})

COLUMN_CHECK_PATH = OUTPUT_DIR / "column_check_2018_2019_floors_3_4_5_7.csv"
column_check.to_csv(COLUMN_CHECK_PATH, index=False)

print("Reference file:", reference_file)
print("Reference column count:", len(reference_columns))
print("Column check saved to:", COLUMN_CHECK_PATH)

if not column_check["same_columns_and_order_as_reference"].all():
    mismatched = column_check.loc[~column_check["same_columns_and_order_as_reference"], "file_name"].tolist()
    raise ValueError(f"Column mismatch found in: {mismatched}")

column_check


## 4. Show the 30 columns used in the merge


In [ ]:
pd.DataFrame({"column_index": range(1, len(reference_columns) + 1), "column_name": reference_columns})


## 5. Count rows before merging


In [ ]:
row_counts = []
for path in source_files:
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        rows = sum(1 for _ in handle) - 1
    year = int(re.search(r"(20\d{2})", path.name).group(1))
    floor = int(re.search(r"Floor(\d+)", path.name).group(1))
    row_counts.append({"year": year, "floor": floor, "file_name": path.name, "row_count": rows})

row_count_table = pd.DataFrame(row_counts)
ROW_COUNT_PATH = OUTPUT_DIR / "row_counts_2018_2019_floors_3_4_5_7.csv"
row_count_table.to_csv(ROW_COUNT_PATH, index=False)

print("Expected merged rows:", f"{row_count_table['row_count'].sum():,}")
print("Row count summary saved to:", ROW_COUNT_PATH)

row_count_table


## 6. Merge the selected CSV files

This section reads each selected raw file, standardizes the `Date` column to datetime, adds traceability columns, and vertically concatenates the tables.


In [ ]:
merged_parts = []

for path in source_files:
    year = int(re.search(r"(20\d{2})", path.name).group(1))
    floor = int(re.search(r"Floor(\d+)", path.name).group(1))

    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df.insert(0, "source_file", path.name)
    df.insert(0, "floor", floor)
    df.insert(0, "year", year)
    merged_parts.append(df)

combined = pd.concat(merged_parts, ignore_index=True)
combined = combined.sort_values(["year", "floor", "Date"]).reset_index(drop=True)

combined.shape


## 7. Validate the merged table


In [ ]:
validation_summary = {
    "rows": len(combined),
    "columns": combined.shape[1],
    "expected_rows_from_sources": int(row_count_table["row_count"].sum()),
    "invalid_datetime_rows": int(combined["Date"].isna().sum()),
    "duplicate_year_floor_date_rows": int(combined.duplicated(["year", "floor", "Date"]).sum()),
    "min_date": combined["Date"].min(),
    "max_date": combined["Date"].max(),
}

validation_summary_df = pd.DataFrame([validation_summary])
validation_summary_df


## 8. Save the merged output


In [ ]:
OUTPUT_FILE = OUTPUT_DIR / "combined_wide_2018_2019_floors_3_4_5_7.csv.gz"
PREVIEW_FILE = OUTPUT_DIR / "combined_wide_2018_2019_floors_3_4_5_7_preview.csv"
SUMMARY_FILE = OUTPUT_DIR / "merge_summary_2018_2019_floors_3_4_5_7.csv"

combined.to_csv(OUTPUT_FILE, index=False, compression="gzip")
combined.head(1000).to_csv(PREVIEW_FILE, index=False)
validation_summary_df.to_csv(SUMMARY_FILE, index=False)

print("Merged output saved to:", OUTPUT_FILE)
print("Preview saved to:", PREVIEW_FILE)
print("Merge summary saved to:", SUMMARY_FILE)
print("Rows:", f"{len(combined):,}")
print("Columns:", combined.shape[1])


## 9. Quick review


In [ ]:
display(combined.head())
display(combined.groupby(["year", "floor"]).size().reset_index(name="rows"))
